In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. LLM-Based Trading Insight Generation

In [2]:
env_path = Path("../data/env/.env")

load_dotenv(env_path)

print("GEMINI_API_KEY configured:", bool(os.getenv("GEMINI_API_KEY")))

GEMINI_API_KEY configured: True


In [3]:
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY is missing from .env"

llm = FreeLLM(
    [GoogleAIStudio(api_key)],
    strategy="priority",
    hedge=False,
    max_attempts=1,
)

print("Provider configured: Google AI Studio")
print("API key loaded:", bool(api_key))

Provider configured: Google AI Studio
API key loaded: True


In [4]:
print("GoogleAIStudio methods:")
print([
    name for name in dir(GoogleAIStudio)
    if not name.startswith("_")
])

print("\nFreeLLM discovery-related methods:")
print([
    name for name in dir(llm)
    if "discover" in name.lower() or "model" in name.lower()
])

GoogleAIStudio methods:
['DEFAULT_MODELS', 'TIERS', 'adapt_payload', 'auth_headers', 'avg_latency', 'base_url', 'bench_model', 'capacity', 'chat_path', 'discovery_url', 'expected_latency', 'headers', 'keyless', 'knows_model', 'model_ready', 'model_wait', 'models_path', 'name', 'parse_response', 'rate_limit_scope', 'resolve_models', 'transient_scope', 'url']

FreeLLM discovery-related methods:
['_discovery_done', '_ensure_discovered', 'refresh_models']


In [5]:
print("GoogleAIStudio default models:")
print(GoogleAIStudio.DEFAULT_MODELS)

GoogleAIStudio default models:
[ModelSpec(id='gemini-3.5-flash-lite', tags=('chat', 'fast', 'small', 'tools', 'vision'), ctx=1048576, free=True, priority=0), ModelSpec(id='gemini-3.1-flash-lite', tags=('chat', 'fast', 'small', 'tools', 'vision'), ctx=1048576, free=True, priority=0), ModelSpec(id='gemini-flash-lite-latest', tags=('chat', 'fast', 'small', 'tools', 'vision'), ctx=1048576, free=True, priority=0), ModelSpec(id='gemini-3.5-flash', tags=('chat', 'large', 'tools', 'vision', 'reasoning'), ctx=1048576, free=True, priority=0), ModelSpec(id='gemini-3.7-flash', tags=('chat', 'large', 'tools', 'vision', 'reasoning'), ctx=1048576, free=True, priority=0), ModelSpec(id='gemini-flash-latest', tags=('chat', 'large', 'tools', 'vision', 'reasoning'), ctx=1048576, free=True, priority=0)]


In [6]:
google_provider = llm.providers[0]

print("Configured models:")
for model_spec in google_provider.models:
    print(model_spec.id)

Configured models:
gemini-3.5-flash-lite
gemini-3.1-flash-lite
gemini-flash-lite-latest
gemini-3.5-flash
gemini-3.7-flash
gemini-flash-latest


In [7]:
response = llm.chat(
    [
        {
            "role": "user",
            "content": "Reply with exactly: Gemini connection successful."
        }
    ],
    model="gemini-3.5-flash",
)

print("Response:", response.text)
print("Provider:", response.provider)
print("Model:", response.model)

INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Response: Gemini connection successful.
Provider: google
Model: gemini-3.5-flash


In [ ]:
# File paths
prediction_path = Path("../data/lstm/processed/bitcoin_daily_predictions.csv")
news_path = Path("../data/news/processed/bitcoin_news_retrieved_top10.csv")

# Load saved data
predictions_df = pd.read_csv(prediction_path)
news_df = pd.read_csv(news_path)

print("Predictions shape:", predictions_df.shape)
print("News shape:", news_df.shape)

print("\nLatest prediction:")
display(
    predictions_df
    .sort_values("forecast_date")
    .tail(1)
)

print("\nRetrieved Top 10 news:")
display(
    news_df[
        ["similarity_rank", "title", "source", "published_at_utc", "link"]
    ]
)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


Native API: SUCCESS
Native API


In [9]:
# Get the latest forecast
latest_prediction = (
    predictions_df
    .sort_values("forecast_date")
    .iloc[-1]
)

forecast_context = f"""
Forecast date: {latest_prediction['forecast_date']}
Prediction created at: {latest_prediction['prediction_created_at']}
Predicted Bitcoin closing price: {latest_prediction['predicted_close']:.2f} USD
Actual closing price: Not available yet
Prediction error: Not available yet
"""

# Prepare the retrieved Top 10 news
news_context = "\n\n".join(
    [
        (
            f"News {row['similarity_rank']}\n"
            f"Title: {row['title']}\n"
            f"Source: {row['source']}\n"
            f"Published at (UTC): {row['published_at_utc']}\n"
            f"Summary: {row['summary']}\n"
            f"Content: {row['content']}\n"
            f"Link: {row['link']}"
        )
        for _, row in news_df.sort_values("similarity_rank").iterrows()
    ]
)

print("Forecast context:")
print(forecast_context)

print("\nNumber of news articles:", len(news_df))
print("News context prepared:", bool(news_context))
print("News context characters:", len(news_context))

Forecast context:

Forecast date: 2026-10-12
Prediction created at: 2026-10-12 13:49:25
Predicted Bitcoin closing price: 81637.73 USD
Actual closing price: Not available yet
Prediction error: Not available yet


Number of news articles: 10
News context prepared: True
News context characters: 40482


In [10]:
print("Content status distribution:")
print(news_df["content_status"].value_counts(dropna=False))

print("\nText lengths by article:")
for _, row in news_df.sort_values("similarity_rank").iterrows():
    print(
        f"Rank {row['similarity_rank']}: "
        f"title={len(str(row['title']))}, "
        f"summary={len(str(row['summary']))}, "
        f"content={len(str(row['content']))}"
    )

Content status distribution:
content_status
summary_and_content    9
content_only           1
Name: count, dtype: int64

Text lengths by article:
Rank 1: title=67, summary=158, content=3049
Rank 2: title=78, summary=224, content=6860
Rank 3: title=116, summary=299, content=4042
Rank 4: title=82, summary=205, content=3738
Rank 5: title=75, summary=133, content=2232
Rank 6: title=69, summary=185, content=3588
Rank 7: title=84, summary=3, content=100
Rank 8: title=86, summary=157, content=3042
Rank 9: title=48, summary=118, content=6585
Rank 10: title=76, summary=201, content=2620


In [11]:
news_context = "\n\n".join(
    [
        (
            f"News {row['similarity_rank']}\n"
            f"Title: {row['title']}\n"
            f"Source: {row['source']}\n"
            f"Published at (UTC): {row['published_at_utc']}\n"
            f"Article content: {row['content']}\n"
            f"Source URL: {row['link']}"
        )
        for _, row in news_df.sort_values("similarity_rank").iterrows()
    ]
)

print("Number of news articles:", len(news_df))
print("News context characters:", len(news_context))
print("\nFirst news preview:")
print(news_context[:1000])

Number of news articles: 10
News context characters: 38839

First news preview:
News 1
Title: Bitcoin Could Be Entering a New Bull Market Despite Latest Sell-Off
Source: BeInCrypto
Published at (UTC): 2026-10-11 19:22:47+00:00
Article content: Despite the market dip and liquidations over the past week, Bitcoin has seemingly entered a bull market, as the analysis of its 50-week moving average shows.

BloFin Research argues that Bitcoin's traditional four-year boom-and-bust cycle is gradually losing strength. Future rallies could deliver smaller gains, followed by shorter and less severe downturns.

The strongest evidence comes from Bitcoin's recovery above its 50-week moving average, a technical indicator that previously signaled bull-market transitions in 2015, 2019 and 2023.

Bitcoin reclaimed this level near $78,800 in September. It subsequently traded above $86,000 in early October.

Meanwhile, Bitcoin's MVRV ratio, which compares its current market value with the average price inve

In [12]:
system_prompt = """
You are a financial news interpretation assistant for a Bitcoin forecasting system.

Your role is to interpret the LSTM forecast using the supplied financial news.
You must not generate or modify the numerical forecast.

Follow these rules:
1. Treat the LSTM forecast as the primary quantitative result.
2. Use the retrieved news only as market context. Do not claim that the news
   directly caused the LSTM prediction.
3. Use only facts supported by the supplied forecast and news.
4. Do not invent events, prices, statistics, sources, or explanations.
5. Distinguish reported facts from interpretations and potential risks.
6. If the evidence is insufficient or conflicting, state the uncertainty.
7. Do not claim that the model has a calibrated probability of being correct.
8. Preserve the source name and URL when referring to a news article.
9. Do not provide guaranteed returns or present the output as financial advice.

Generate exactly these four sections:

1. Forecasting Result
   - State the forecast date and predicted Bitcoin closing price.
   - Do not imply that the forecast is the actual market price.

2. Confidence
   - Describe the limitations and uncertainty of the forecast in words.
   - Do not invent a numerical confidence score or probability.

3. Key Market News
   - Summarize the most relevant retrieved news.
   - Explain how the news provides context, without treating it as the
     direct cause of the LSTM prediction.
   - Cite the relevant source and URL.

4. Potential Risks
   - Identify risks supported by the supplied forecast and news.
   - Clearly distinguish reported facts from possible implications.
   - State when the available evidence is insufficient.

Write clearly and concisely.
"""

user_prompt = f"""
Interpret the following Bitcoin forecast using the retrieved news.

LSTM forecast:
{forecast_context}

Retrieved news:
{news_context}
"""

print("System prompt prepared:", bool(system_prompt))
print("User prompt prepared:", bool(user_prompt))
print("User prompt characters:", len(user_prompt))
print("\nSystem prompt preview:")
print(system_prompt)

System prompt prepared: True
User prompt prepared: True
User prompt characters: 39134

System prompt preview:

You are a financial news interpretation assistant for a Bitcoin forecasting system.

Your role is to interpret the LSTM forecast using the supplied financial news.
You must not generate or modify the numerical forecast.

Follow these rules:
1. Treat the LSTM forecast as the primary quantitative result.
2. Use the retrieved news only as market context. Do not claim that the news
   directly caused the LSTM prediction.
3. Use only facts supported by the supplied forecast and news.
4. Do not invent events, prices, statistics, sources, or explanations.
5. Distinguish reported facts from interpretations and potential risks.
6. If the evidence is insufficient or conflicting, state the uncertainty.
7. Do not claim that the model has a calibrated probability of being correct.
8. Preserve the source name and URL when referring to a news article.
9. Do not provide guaranteed returns or 

In [13]:
print("System prompt characters:", len(system_prompt))
print("Forecast context characters:", len(forecast_context))
print("News context characters:", len(news_context))
print("Total user prompt characters:", len(user_prompt))

print("\nNews content lengths:")
print(
    news_df.assign(
        content_length=news_df["content"].fillna("").astype(str).str.len()
    )[["similarity_rank", "title", "content_length"]]
)

System prompt characters: 1733
Forecast context characters: 192
News context characters: 38839
Total user prompt characters: 39134

News content lengths:
   similarity_rank                                              title  \
0                1  Bitcoin Could Be Entering a New Bull Market De...   
1                2  AI Price Prediction: Insights from Grok on Bit...   
2                3  Bitcoin and ethereum prices today, Friday, Oct...   
3                4  Bitcoin rebounds above $83,000 after weekly sl...   
4                5  Bitcoin Rebounds After Rough Week, But Traders...   
5                6  Bitcoin struggles above $83,000 a year after $...   
6                7  Bitcoin, Strategy, and Crypto Stocks Rebound a...   
7                8  Bitcoin Tests Average Buyer’s Break-Even As ET...   
8                9   Weekly Wrap: Rising Treasury Yields Sink Bitcoin   
9               10  Bitcoin holds ground above $82k, heads for wee...   

   content_length  
0            3049  
1 

In [34]:
news_context_full = "\n\n".join(
    (
        f"News {row['similarity_rank']}\n"
        f"Title: {row['title']}\n"
        f"Source: {row['source']}\n"
        f"Published at (UTC): {row['published_at_utc']}\n"
        f"Content status: {row['content_status']}\n"
        f"Saved article content:\n{row['content']}\n"
        f"Source URL: {row['link']}"
    )
    for _, row in news_df.sort_values("similarity_rank").iterrows()
)

user_prompt_full = f"""
Interpret the following Bitcoin forecast using the saved news content.

LSTM forecast:
{forecast_context}

Retrieved news:
{news_context_full}
"""

print("Number of news articles:", len(news_df))
print("News context characters:", len(news_context_full))
print("Total user prompt characters:", len(user_prompt_full))
print("API called: No")

Number of news articles: 10
News context characters: 39252
Total user prompt characters: 39551
API called: No


In [35]:
try:
    response_full = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_prompt_full,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=2048,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            ),
        ),
    )

    if response_full.text:
        llm_insight = response_full.text
        print("Model: gemini-3.5-flash")
        print("Finish reason:",
              response_full.candidates[0].finish_reason
              if response_full.candidates else "No candidate")
        print("\nGenerated Trading Insights:\n")
        print(llm_insight)
    else:
        print("No text returned.")
        print("Candidates:", response_full.candidates)
        print("Usage:", response_full.usage_metadata)

except Exception as e:
    print("Request failed:", type(e).__name__)
    print(str(e))
    print("Do not retry automatically.")

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"


Request failed: ServerError
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Do not retry automatically.


In [ ]:
# Prepare Batch 1: news ranks 1–5
batch1_df = (
    news_df.sort_values("similarity_rank")
    .head(5)
    .copy()
)

batch1_news_context = "\n\n".join(
    (
        f"News {row['similarity_rank']}\n"
        f"Title: {row['title']}\n"
        f"Source: {row['source']}\n"
        f"Published at (UTC): {row['published_at_utc']}\n"
        f"Content status: {row['content_status']}\n"
        f"Saved article content:\n{row['content']}\n"
        f"Source URL: {row['link']}"
    )
    for _, row in batch1_df.iterrows()
)

batch1_user_prompt = f"""
Interpret the following Bitcoin forecast using the saved news content.

LSTM forecast:
{forecast_context}

Retrieved news (Batch 1 of 2, ranks 1-5):
{batch1_news_context}

Use only the supplied forecast and news.
Do not invent facts or claim that news directly caused the LSTM forecast.
"""

print("Batch:", "1 of 2")
print("News ranks:", batch1_df["similarity_rank"].tolist())
print("Number of articles:", len(batch1_df))
print("News context characters:", len(batch1_news_context))
print("Total user prompt characters:", len(batch1_user_prompt))
print("API called: No")

Batch: 1 of 2
News ranks: [1, 2, 3, 4, 5]
Number of articles: 5
News context characters: 21698
Total user prompt characters: 22139
API called: No


In [37]:
# Prepare Batch 2: news ranks 6–10
batch2_df = (
    news_df.sort_values("similarity_rank")
    .iloc[5:10]
    .copy()
)

batch2_news_context = "\n\n".join(
    (
        f"News {row['similarity_rank']}\n"
        f"Title: {row['title']}\n"
        f"Source: {row['source']}\n"
        f"Published at (UTC): {row['published_at_utc']}\n"
        f"Content status: {row['content_status']}\n"
        f"Saved article content:\n{row['content']}\n"
        f"Source URL: {row['link']}"
    )
    for _, row in batch2_df.iterrows()
)

batch2_user_prompt = f"""
Interpret the following Bitcoin forecast using the saved news content.

LSTM forecast:
{forecast_context}

Retrieved news (Batch 2 of 2, ranks 6–10):
{batch2_news_context}

Use only the supplied forecast and news.
Do not invent facts or claim that news directly caused the LSTM forecast.
"""

print("Batch:", "2 of 2")
print("News ranks:", batch2_df["similarity_rank"].tolist())
print("Number of articles:", len(batch2_df))
print("News context characters:", len(batch2_news_context))
print("Total user prompt characters:", len(batch2_user_prompt))
print("API called: No")

Batch: 2 of 2
News ranks: [6, 7, 8, 9, 10]
Number of articles: 5
News context characters: 17552
Total user prompt characters: 17994
API called: No


In [ ]:
try:
    response_batch1 = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=batch1_user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=2048,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            ),
        ),
    )

    print("Model: gemini-3.5-flash")
    print("Batch: 1 (News ranks 1–5)")

    if response_batch1.candidates:
        print("Finish reason:",
              response_batch1.candidates[0].finish_reason)

    if response_batch1.text:
        batch1_insight = response_batch1.text
        print("\nGenerated Insights:\n")
        print(batch1_insight)
    else:
        print("\nNo text returned.")
        print("Candidates:", response_batch1.candidates)
        print("Usage:", response_batch1.usage_metadata)

except Exception as e:
    print("Request failed:", type(e).__name__)
    print(str(e))
    print("Do not retry automatically.")

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


Model: gemini-3.5-flash
Batch: 1 (News ranks 1–5)
Finish reason: FinishReason.STOP

Generated Insights:

### 1. Forecasting Result
* **Forecast Date:** October 12, 2026
* **Predicted Bitcoin Closing Price:** $81,637.73 USD
* *Note:* This figure represents the quantitative prediction generated by the LSTM model and should not be interpreted as the actual live or final closing market price.

### 2. Confidence
The provided LSTM prediction does not include a numerical confidence score or a calibrated probability of accuracy. Financial time-series forecasts, including LSTM models, inherent significant limitations due to rapid market shifts, macro volatility, and liquidity changes. Consequently, this single price point carries natural uncertainty and should be viewed as a theoretical quantitative baseline rather than a guaranteed outcome.

### 3. Key Market News
The retrieved market news offers broader contextual environment surrounding the October 12 forecast date:
* **Recent Price Levels a

In [40]:
try:
    response_batch2 = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=batch2_user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=2048,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            ),
        ),
    )

    print("Model: gemini-3.5-flash")
    print("Batch: 2 (News ranks 6-10)")

    if response_batch2.candidates:
        print("Finish reason:",
              response_batch2.candidates[0].finish_reason)

    if response_batch2.text:
        batch2_insight = response_batch2.text
        print("\nGenerated Insights:\n")
        print(batch2_insight)
    else:
        print("\nNo text returned.")
        print("Candidates:", response_batch2.candidates)
        print("Usage:", response_batch2.usage_metadata)

except Exception as e:
    print("Request failed:", type(e).__name__)
    print(str(e))
    print("Do not retry automatically.")

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


Model: gemini-3.5-flash
Batch: 2 (News ranks 6-10)
Finish reason: FinishReason.STOP

Generated Insights:

### 1. Forecasting Result
* **Forecast Date:** 2026-10-12
* **Predicted Bitcoin Closing Price:** 81,637.73 USD

*Note: This predicted value is an output generated by an quantitative LSTM model and represents an estimate, not a guaranteed or actual market closing price.*

---

### 2. Confidence
The model’s output carries inherent uncertainty due to the volatile nature of cryptocurrency markets. Quantitative forecasts rely strictly on historical patterns and cannot account for unexpected macroeconomic shifts, geopolitical events, or sudden swings in market sentiment. Furthermore, the provided output does not establish a calibrated probability or statistical certainty of being accurate, and actual closing prices may diverge significantly from the predicted figure.

---

### 3. Key Market News
The retrieved news provides surrounding macroeconomic and market context for the period leadi

In [42]:
from pathlib import Path

# Output directory
output_dir = Path("../data/llm")
output_dir.mkdir(parents=True, exist_ok=True)

# Check that both generated results exist in memory
assert "batch1_insight" in globals(), "Batch 1 result not found."
assert "batch2_insight" in globals(), "Batch 2 result not found."

# Save individual batch results
batch1_path = output_dir / "bitcoin_llm_insights_batch1.txt"
batch2_path = output_dir / "bitcoin_llm_insights_batch2.txt"

batch1_path.write_text(batch1_insight, encoding="utf-8")
batch2_path.write_text(batch2_insight, encoding="utf-8")

# Merge both outputs while preserving their original content
merged_insight = (
    "BITCOIN LLM INSIGHTS\n"
    "Generated using gemini-3.5-flash\n"
    "News Batch 1: Similarity ranks 1-5\n\n"
    + batch1_insight.strip()
    + "\n\n"
    + "=" * 80
    + "\n\nNews Batch 2: Similarity ranks 6-10\n\n"
    + batch2_insight.strip()
    + "\n"
)

merged_path = output_dir / "bitcoin_llm_insights_merged.txt"
merged_path.write_text(merged_insight, encoding="utf-8")

print("Batch 1 saved:", batch1_path.resolve())
print("Batch 2 saved:", batch2_path.resolve())
print("Merged file saved:", merged_path.resolve())
print("Batch 1 characters:", len(batch1_insight))
print("Batch 2 characters:", len(batch2_insight))
print("Merged characters:", len(merged_insight))

Batch 1 saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_batch1.txt
Batch 2 saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_batch2.txt
Merged file saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_merged.txt
Batch 1 characters: 3775
Batch 2 characters: 3618
Merged characters: 7605


In [43]:
merged_path = Path("../data/llm/bitcoin_llm_insights_merged.txt")
merged_text = merged_path.read_text(encoding="utf-8")

required_sections = [
    "Forecasting Result",
    "Confidence",
    "Key Market News",
    "Potential Risks",
]

print("Merged file exists:", merged_path.exists())
print("Merged characters:", len(merged_text))

print("\nSection occurrence counts:")
for section in required_sections:
    count = len(re.findall(re.escape(section), merged_text))
    print(f"{section}: {count}")

print("\nBatch markers:")
print("Batch 1 marker:", "News Batch 1" in merged_text)
print("Batch 2 marker:", "News Batch 2" in merged_text)

Merged file exists: True
Merged characters: 7605

Section occurrence counts:
Forecasting Result: 2
Confidence: 2
Key Market News: 2
Potential Risks: 2

Batch markers:
Batch 1 marker: True
Batch 2 marker: True


In [44]:
for batch_name, text in [
    ("Batch 1", batch1_insight),
    ("Batch 2", batch2_insight),
]:
    print(f"\n--- {batch_name} ---")

    for match in re.finditer(
        r"(?m)^#{1,3}\s*\d*\.?\s*(Forecasting Result|Confidence|Key Market News|Potential Risks)\s*$",
        text,
    ):
        print(match.group(1), "at character", match.start())


--- Batch 1 ---
Forecasting Result at character 0
Confidence at character 288
Key Market News at character 741
Potential Risks at character 2591

--- Batch 2 ---
Forecasting Result at character 0
Confidence at character 277
Key Market News at character 778
Potential Risks at character 2663


In [45]:
for batch_name, text in [
    ("Batch 1", batch1_insight),
    ("Batch 2", batch2_insight),
]:
    print(f"\n{'=' * 30}\n{batch_name}\n{'=' * 30}\n")
    print(text)


Batch 1

### 1. Forecasting Result
* **Forecast Date:** October 12, 2026
* **Predicted Bitcoin Closing Price:** $81,637.73 USD
* *Note:* This figure represents the quantitative prediction generated by the LSTM model and should not be interpreted as the actual live or final closing market price.

### 2. Confidence
The provided LSTM prediction does not include a numerical confidence score or a calibrated probability of accuracy. Financial time-series forecasts, including LSTM models, inherent significant limitations due to rapid market shifts, macro volatility, and liquidity changes. Consequently, this single price point carries natural uncertainty and should be viewed as a theoretical quantitative baseline rather than a guaranteed outcome.

### 3. Key Market News
The retrieved market news offers broader contextual environment surrounding the October 12 forecast date:
* **Recent Price Levels and Trends:** Bitcoin has recently experienced pullbacks after reaching near $87,000 on October 

In [46]:
# 1. Define the four required sections
section_names = [
    "Forecasting Result",
    "Confidence",
    "Key Market News",
    "Potential Risks",
]

def extract_sections(text):
    pattern = (
        r"(?m)^#{1,3}\s*\d*\.?\s*"
        r"(Forecasting Result|Confidence|Key Market News|Potential Risks)\s*$"
    )

    matches = list(re.finditer(pattern, text))
    sections = {}

    for i, match in enumerate(matches):
        name = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[name] = text[start:end].strip()

    return sections

# 2. Extract the sections from both saved results in memory
sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

print("Batch 1 sections:", list(sections1.keys()))
print("Batch 2 sections:", list(sections2.keys()))

# 3. Check that all required sections exist before creating the file
assert all(name in sections1 for name in section_names), \
    "Batch 1 is missing a required section."
assert all(name in sections2 for name in section_names), \
    "Batch 2 is missing a required section."

# 4. Build a review draft:
# Keep one forecast and confidence section.
# Preserve both batches' news and risk text for manual review.
final_draft = []

final_draft.append("## 1. Forecasting Result\n")
final_draft.append(sections1["Forecasting Result"])

final_draft.append("\n\n## 2. Confidence\n")
final_draft.append(sections1["Confidence"])

final_draft.append("\n\n## 3. Key Market News\n")
final_draft.append("### Batch 1\n" + sections1["Key Market News"])
final_draft.append("\n\n### Batch 2\n" + sections2["Key Market News"])

final_draft.append("\n\n## 4. Potential Risks\n")
final_draft.append("### Batch 1\n" + sections1["Potential Risks"])
final_draft.append("\n\n### Batch 2\n" + sections2["Potential Risks"])

final_text = "\n".join(final_draft).strip() + "\n"

# 5. Save as a new file without overwriting existing outputs
output_dir = Path("../data/llm")
output_dir.mkdir(parents=True, exist_ok=True)

final_path = output_dir / "bitcoin_llm_insights_final.txt"
final_path.write_text(final_text, encoding="utf-8")

print("\nReview draft saved:", final_path.resolve())
print("Characters:", len(final_text))
print("Original batch files and merged file were not overwritten.")


Batch 1 sections: ['Forecasting Result', 'Confidence', 'Key Market News', 'Potential Risks']
Batch 2 sections: ['Forecasting Result', 'Confidence', 'Key Market News', 'Potential Risks']

Review draft saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_final.txt
Characters: 6625
Original batch files and merged file were not overwritten.


In [48]:
# Build the source list for Batch 2 from the saved Top 10 news CSV
batch2_sources = (
    news_df.sort_values("similarity_rank")
    .iloc[5:10]
)

batch2_source_text = "\n".join(
    f"- News {row['similarity_rank']}: {row['title']}\n"
    f"  Source: {row['source']}\n"
    f"  URL: {row['link']}"
    for _, row in batch2_sources.iterrows()
)

print("Sources for Batch 2 (ranks 6–10):\n")
print(batch2_source_text)

Sources for Batch 2 (ranks 6–10):

- News 6: Bitcoin struggles above $83,000 a year after $19 billion crypto crash
  Source: Investing.com
  URL: https://finance.yahoo.com/markets/crypto/articles/bitcoin-struggles-above-83-000-055426754.html
- News 7: Bitcoin, Strategy, and Crypto Stocks Rebound as Trump Signal Cools Iran Attack Fears
  Source: Barrons.com
  URL: https://finance.yahoo.com/m/fdafdedd-9b80-3ca6-b5dc-bc31417773c8/bitcoin-strategy-and-crypto.html
- News 8: Bitcoin Tests Average Buyer’s Break-Even As ETF Outflows Clash With Wall Street Buying
  Source: Stocktwits
  URL: https://finance.yahoo.com/markets/crypto/articles/bitcoin-tests-average-buyer-break-173425678.html
- News 9: Weekly Wrap: Rising Treasury Yields Sink Bitcoin
  Source: CryptoProwl
  URL: https://finance.yahoo.com/markets/crypto/articles/weekly-wrap-rising-treasury-yields-205200729.html
- News 10: Bitcoin holds ground above $82k, heads for weekly losses amid yield pressure
  Source: Investing.com
  URL: https

In [49]:
output_dir = Path("../data/llm")
final_path = output_dir / "bitcoin_llm_insights_final.txt"

# Build the source list directly from the saved Top 10 news data
batch2_sources = (
    news_df.sort_values("similarity_rank")
    .iloc[5:10]
)

batch2_source_text = "\n".join(
    f"- News {row['similarity_rank']}: {row['title']}\n"
    f"  Source: {row['source']}\n"
    f"  URL: {row['link']}"
    for _, row in batch2_sources.iterrows()
)

# Read the current final draft
final_text = final_path.read_text(encoding="utf-8")

# Append the missing source list without duplicating it
source_heading = "### Sources — Batch 2 (News 6–10)"

if source_heading not in final_text:
    final_text = (
        final_text.rstrip()
        + "\n\n"
        + source_heading
        + "\n\n"
        + batch2_source_text
        + "\n"
    )

final_path.write_text(final_text, encoding="utf-8")

print("Updated:", final_path.resolve())
print("Batch 2 sources added:", len(batch2_sources))
print("Final report characters:", len(final_text))
print("API called: No")

Updated: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_final.txt
Batch 2 sources added: 5
Final report characters: 7717
API called: No


In [50]:
final_path = Path("../data/llm/bitcoin_llm_insights_final.txt")
final_text = final_path.read_text(encoding="utf-8")

source_heading = "### Sources — Batch 2 (News 6–10)"

# Extract the existing Batch 2 source list
if source_heading not in final_text:
    raise ValueError("Batch 2 source list was not found.")

before_sources, after_heading = final_text.split(source_heading, 1)

# The source list currently extends to the end of the file
batch2_source_block = after_heading.strip()

# Remove the misplaced source list
clean_text = before_sources.rstrip()

# Insert it immediately before the Potential Risks section
risks_heading = "## 4. Potential Risks"

if risks_heading not in clean_text:
    raise ValueError("Potential Risks heading was not found.")

clean_text = clean_text.replace(
    risks_heading,
    source_heading + "\n\n" + batch2_source_block + "\n\n" + risks_heading,
    1,
)

final_path.write_text(clean_text.rstrip() + "\n", encoding="utf-8")

print("Updated:", final_path.resolve())
print("Batch 2 source list moved before Potential Risks.")
print("API called: No")

Updated: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_final.txt
Batch 2 source list moved before Potential Risks.
API called: No


In [52]:
final_path = Path("../data/llm/bitcoin_llm_insights_final.txt")
final_text = final_path.read_text(encoding="utf-8")

# Check the order of the main sections and Batch 2 sources
markers = [
    "## 1. Forecasting Result",
    "## 2. Confidence",
    "## 3. Key Market News",
    "### Sources — Batch 2 (News 6–10)",
    "## 4. Potential Risks",
]

positions = {marker: final_text.find(marker) for marker in markers}

for marker, position in positions.items():
    print(f"{marker}: {'FOUND' if position >= 0 else 'MISSING'}")

if all(position >= 0 for position in positions.values()):
    print(
        "\nCorrect order:",
        list(positions.values()) == sorted(positions.values())
    )

print("\nBatch 2 source entries:", final_text.count("- News "))
print("Final report characters:", len(final_text))
print("API called: No")

## 1. Forecasting Result: FOUND
## 2. Confidence: FOUND
## 3. Key Market News: FOUND
### Sources — Batch 2 (News 6–10): FOUND
## 4. Potential Risks: FOUND

Correct order: True

Batch 2 source entries: 5
Final report characters: 7717
API called: No


In [53]:
final_path = Path("../data/llm/bitcoin_llm_insights_final.txt")
final_text = final_path.read_text(encoding="utf-8")

# Verify that every saved Top 10 news URL appears in the final report
missing_sources = []

for _, row in news_df.sort_values("similarity_rank").iterrows():
    url = str(row["link"])

    if url not in final_text:
        missing_sources.append({
            "rank": int(row["similarity_rank"]),
            "source": row["source"],
            "title": row["title"],
        })

print("Total news articles:", len(news_df))
print("Missing source URLs:", len(missing_sources))

if missing_sources:
    print("\nSources still missing:")
    for item in missing_sources:
        print(
            f"News {item['rank']} | "
            f"{item['source']} | "
            f"{item['title']}"
        )
else:
    print("All 10 news URLs are present in the final report.")

print("\nAPI called: No")

Total news articles: 10
Missing source URLs: 0
All 10 news URLs are present in the final report.

API called: No


In [54]:
# Compare the main topics covered by both batches
for batch_name, text in [
    ("Batch 1", batch1_insight),
    ("Batch 2", batch2_insight),
]:
    sections = extract_sections(text)

    print(f"\n{'=' * 20} {batch_name} {'=' * 20}")

    print("\nKey Market News:")
    print(sections["Key Market News"])

    print("\nPotential Risks:")
    print(sections["Potential Risks"])


==================== Batch 1 ====================

Key Market News:
The retrieved market news offers broader contextual environment surrounding the October 12 forecast date:
* **Recent Price Levels and Trends:** Bitcoin has recently experienced pullbacks after reaching near $87,000 on October 2, dipping toward the $80,000 mark before stabilizing back in the $82,000–$83,000 range between October 9 and October 11 (*24/7 Wall St.*, *Investing.com*, *Decrypt*).
* **Institutional ETF Outflows:** Spot Bitcoin ETFs faced substantial net outflows—including over $679 million to $731 million across mid-October sessions—reflecting short-term institutional caution despite strong underlying long-term holdings (*BeInCrypto*, *24/7 Wall St.*, *Investing.com*, *Decrypt*).
* **Long-Term Structural Signals:** Research from BloFin indicates Bitcoin reclaimed its 50-week moving average (around $78,800) in September, with long-term holders controlling approximately 83% of the circulating supply (*BeInCryp

In [55]:
sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

print("========== BATCH 1: KEY MARKET NEWS ==========")
print(sections1["Key Market News"])

print("\n========== BATCH 2: KEY MARKET NEWS ==========")
print(sections2["Key Market News"])

print("\n========== BATCH 1: POTENTIAL RISKS ==========")
print(sections1["Potential Risks"])

print("\n========== BATCH 2: POTENTIAL RISKS ==========")
print(sections2["Potential Risks"])

========== BATCH 1: KEY MARKET NEWS ==========
The retrieved market news offers broader contextual environment surrounding the October 12 forecast date:
* **Recent Price Levels and Trends:** Bitcoin has recently experienced pullbacks after reaching near $87,000 on October 2, dipping toward the $80,000 mark before stabilizing back in the $82,000–$83,000 range between October 9 and October 11 (*24/7 Wall St.*, *Investing.com*, *Decrypt*).
* **Institutional ETF Outflows:** Spot Bitcoin ETFs faced substantial net outflows—including over $679 million to $731 million across mid-October sessions—reflecting short-term institutional caution despite strong underlying long-term holdings (*BeInCrypto*, *24/7 Wall St.*, *Investing.com*, *Decrypt*).
* **Long-Term Structural Signals:** Research from BloFin indicates Bitcoin reclaimed its 50-week moving average (around $78,800) in September, with long-term holders controlling approximately 83% of the circulating supply (*BeInCrypto*).
* **AI & Options

In [56]:
output_dir = Path("../data/llm")
final_path = output_dir / "bitcoin_llm_insights_final.txt"

# Extract the four sections from the two successful model responses
sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

# Use Batch 1 for the shared forecast and confidence sections
forecast_section = sections1["Forecasting Result"]
confidence_section = sections1["Confidence"]

# Keep both news analyses, without discarding unique information
news_section = (
    "The following points summarise the retrieved news context. "
    "Reported figures have not been independently verified.\n\n"
    "### Market price and technical context\n"
    + sections1["Key Market News"]
    + "\n\n### Macroeconomic, ETF and leverage context\n"
    + sections2["Key Market News"]
)

# Preserve the risk details from both batches for now.
# A later review can remove repeated wording without losing unique risks.
risk_section = (
    "### Risks identified in Batch 1\n"
    + sections1["Potential Risks"]
    + "\n\n### Additional risks identified in Batch 2\n"
    + sections2["Potential Risks"]
)

# Build a complete source list from the original Top 10 CSV
sources_section = "\n".join(
    f"- News {int(row['similarity_rank'])}: {row['title']}\n"
    f"  Source: {row['source']}\n"
    f"  URL: {row['link']}"
    for _, row in news_df.sort_values("similarity_rank").iterrows()
)

unified_report = f"""# Bitcoin LLM Insights

## 1. Forecasting Result

{forecast_section}

## 2. Confidence

{confidence_section}

## 3. Key Market News

{news_section}

## 4. Potential Risks

{risk_section}

## Sources

{sources_section}
"""

# Save to a separate review file first
review_path = output_dir / "bitcoin_llm_insights_unified_review.txt"
review_path.write_text(unified_report, encoding="utf-8")

print("Unified review saved:", review_path.resolve())
print("Characters:", len(unified_report))
print("Sources included:", len(news_df))
print("API called: No")

Unified review saved: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\llm\bitcoin_llm_insights_unified_review.txt
Characters: 9147
Sources included: 10
API called: No
